# 00 — Carga de datos

Localiza primero los JSONL existentes y solo descarga con KaggleHub si no están disponibles.

Si aparece `UnauthenticatedError`, configura `AP2026Q2_DATA_DIR` con la carpeta que contiene `train_data.jsonlines` y `test_data.jsonlines`, o autentica Kaggle con `KAGGLE_API_TOKEN`. Las credenciales y los datos quedan fuera de Git.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Entorno local: no monto Drive.')

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
for p in (ROOT,*ROOT.parents):
    if (p/'src'/'ap2026q2_common.py').exists(): ROOT=p; break
sys.path.insert(0,str(ROOT/'src'))
from ap2026q2_common import *
print('Repo:',ROOT,'Datos:',project_data_dir())

In [ ]:
from pathlib import Path
import os

def _has_competition_files(folder):
    return bool(folder) and (folder / "train_data.jsonlines").exists() and (folder / "test_data.jsonlines").exists()

# Prioridad: variable configurada, carpeta local habitual y Drive de Colab.
configured = os.getenv("AP2026Q2_DATA_DIR")
candidates = [Path(configured).expanduser()] if configured else []
candidates += [Path.home() / "OneDrive" / "Documents" / "TP 3 AP"]
candidates += [Path("/content/drive/MyDrive/Analisis_Predictivo/ap_2026_q_2")]
local_data = next((p for p in candidates if _has_competition_files(p)), None)
if local_data is not None:
    os.environ["AP2026Q2_DATA_DIR"] = str(local_data)
    print("Uso datos locales:", local_data)

raw_dir = download_competition()
train, test, sample = load_raw_data()
print("Train:", train.shape, "Test:", test.shape)
print("Template oficial:", None if sample is None else sample.shape, "| IDs de submission: 1..N si no existe")\nprint("Directorio:", raw_dir)

In [ ]:
display(train.head(3))
display(test.head(3))
if sample is not None: display(sample.head(3))